# MODEL PREDICTION ANALYSIS

The objective of this notebook is to find meaningful insights on the model predictions and performances. We will start by analyzing question by themes, and then analyzing by the type of sources provided in the benchmark.

### Necessary imports

In [4]:
import json
import glob
import os
import pandas as pd
import numpy as np

### Helpers Methods

In [6]:
def load_data(files_pattern):
    data = []
    files = glob.glob(files_pattern)
    print(f"Found {len(files)} files to analyze.")
        
    for file_path in files:
        with open(file_path, 'r') as f:
            try:
                content = json.load(f)
                if isinstance(content, list):
                    items = content
                else:
                    items = [content]
                for i, item in enumerate(items):
                    data.append({
                        "id": i,
                        "question": item.get("question", ""),
                        "correct": item.get("correct", False),
                        "file": os.path.basename(file_path)
                    })
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return pd.DataFrame(data)

def get_stats(column, df):
    summary = df.groupby(column).agg(
        count=("correct", "size"),
        Accuracy=("correct", "mean")
    ).reset_index()
    
    overall = pd.DataFrame([{
        column: "Overall",
        "count": df["correct"].size,
        "Accuracy": df["correct"].mean()
    }])

    final_summary = pd.concat([summary, overall], ignore_index=True)
    return final_summary, df

def print_stats(column, df):
    print(f"=============== Analysis by {column} ===============")
    print(df.to_string(formatters={
        'Accuracy': '{:,.2%}'.format
    }, index=False))

## Theme Analysis 

- Here we try to analyze on which topic does the models tend to have success or to fail

In [9]:
THEME_KEYWORDS = {
    "Wounded, Sick & Dead": ["dead", "body", "bodies", "remains", "wounded", "sick", "medical", "hospital", "ambulance"],
    "Prisoners of War (POW)": ["prisoner", "captured", "detainee", "detention", "pow", "internment"],
    "Civilians": ["civilian", "protected person", "women", "children", "journalist", "humanitarian relief"],
    "Conduct of Hostilities": ["attack", "target", "weapon", "proportionality", "distinction", "military objective", "precautions"],
    "Occupation": ["occupation", "occupied", "occupying power"],
    "Emblems & Signs": ["emblem", "red cross", "red crescent", "flag", "insignia"],
}

def get_theme_by_keyword(question):
    q_lower = question.lower()
    
    for theme, keywords in THEME_KEYWORDS.items():
        if any(k in q_lower for k in keywords):
            return theme
            
    return "General / Other"

def get_themes(df):
    path = "../datasets/law_benchmark_data.json"
    df["Theme"] = ""
    with open(path, 'r') as f:
        try:
            content = json.load(f)
            if isinstance(content, list):
                items = content
            else:
                items = [content]
            for i, item in enumerate(items):
                theme = get_theme_by_keyword(item.get("question",""))
                df.loc[df["id"]==i,"Theme"] = theme
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return df

def analyze_themes(file_path):
    df_raw = load_data(file_path)
    df_themed = get_themes(df_raw)
    COLUMN = "Theme"
    summary, df = get_stats(COLUMN, df_themed)
    print_stats(COLUMN, summary)
    return df

In [10]:
df_all_theme = analyze_themes("predictions/*.json")

Found 19 files to analyze.
=============== Analysis by Theme ===============
                 Theme  count Accuracy
             Civilians   1348   67.95%
Conduct of Hostilities   1284   62.46%
       Emblems & Signs    246   54.07%
       General / Other   4548   63.63%
            Occupation    169   91.12%
Prisoners of War (POW)    562   72.24%
  Wounded, Sick & Dead    662   44.71%
               Overall   8819   63.51%


- Let's observe the impact of RAG on the llama base model

In [12]:
df_llama_theme = analyze_themes("predictions/llama3.1-8B.json")

Found 1 files to analyze.
=============== Analysis by Theme ===============
                 Theme  count Accuracy
             Civilians     72   75.00%
Conduct of Hostilities     68   69.12%
       Emblems & Signs     13   46.15%
       General / Other    244   65.98%
            Occupation      9   66.67%
Prisoners of War (POW)     30   76.67%
  Wounded, Sick & Dead     35   51.43%
               Overall    471   66.88%


In [13]:
df_llama_RAG_theme = analyze_themes("predictions/llama3.1-8B_RAG.json")

Found 1 files to analyze.
=============== Analysis by Theme ===============
                 Theme  count Accuracy
             Civilians     72   72.22%
Conduct of Hostilities     68   61.76%
       Emblems & Signs     13   53.85%
       General / Other    244   65.98%
            Occupation      9  100.00%
Prisoners of War (POW)     30   73.33%
  Wounded, Sick & Dead     35   45.71%
               Overall    471   65.61%


## Source Analysis

- We analyse for which type of sources does the models succeed in general, and the impact of using RAG to provide the IHL rules to the model

In [16]:
def get_source_category(sources):
    """
    Classifies a list of sources into 'Rules', 'Reports', or 'Mixed'.
    """
    if not sources:
        return "Unknown"

    has_rule = False
    has_report = False
    s_lower = str(sources).lower()
    if "rule" in s_lower:
        has_rule = True
    elif "icrc" in s_lower:
        has_report = True

    if has_rule and not has_report:
        return "IHL Rules"
    elif has_report and not has_rule:
        return "ICRC Reports"
    elif has_rule and has_report:
        return "Mixed (Rule + Report)"
    else:
        return "Other"

def get_sources(df):
    path = "../datasets/law_benchmark_data.json"
    df["Source"] = ""
    with open(path, 'r') as f:
        try:
            content = json.load(f)
            if isinstance(content, list):
                items = content
            else:
                items = [content]
            for i, item in enumerate(items):
                source = get_source_category(item.get("source", None))
                df.loc[df["id"]==i,"Source"] = source
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return df

def analyze_sources(file_path):
    df_raw = load_data(file_path)
    df_sourced = get_sources(df_raw)
    COLUMN = "Source"
    summary, df = get_stats(COLUMN, df_sourced)
    print_stats(COLUMN, summary)
    return df

In [17]:
df_all = analyze_sources("predictions/*.json")

Found 19 files to analyze.
=============== Analysis by Source ===============
      Source  count Accuracy
ICRC Reports   3819   85.57%
   IHL Rules   4850   46.80%
     Unknown    150   42.00%
     Overall   8819   63.51%


- Let's observe the impact of RAG on the llama base model

In [19]:
df_llama = analyze_sources("predictions/llama3.1-8B.json")

Found 1 files to analyze.
=============== Analysis by Source ===============
      Source  count Accuracy
ICRC Reports    201   84.08%
   IHL Rules    262   54.20%
     Unknown      8   50.00%
     Overall    471   66.88%


In [20]:
df_llama_RAG = analyze_sources("predictions/llama3.1-8B_RAG.json")

Found 1 files to analyze.
=============== Analysis by Source ===============
      Source  count Accuracy
ICRC Reports    201   88.06%
   IHL Rules    262   48.09%
     Unknown      8   75.00%
     Overall    471   65.61%


- We see that some cells have sources labeled as Unknown, in this case it means that no source ground truth was provided for that question :

In [22]:
# Questions without sources
UNKNOWN_QUESTIONS = [17,21,24,30,74,119,399,431]

In [23]:
df_llama.loc[UNKNOWN_QUESTIONS,"correct"]

17     False
21     False
24     False
30      True
74     False
119     True
399     True
431     True
Name: correct, dtype: bool

In [24]:
df_llama_RAG.loc[UNKNOWN_QUESTIONS,"correct"]

17     False
21     False
24      True
30      True
74      True
119     True
399     True
431     True
Name: correct, dtype: bool